In [12]:
# Check the summary of Delly_BND
import os
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
wgs_file = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Delly/Delly_ANNOT_GTF/ALL_SAMPLES.BND.pairs.tsv"
outdir = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Delly/Delly_ANNOT_GTF/WGS_pair_summary"
os.makedirs(outdir, exist_ok=True)

# Choose one:
# "ordered"   = Gene_ChrA--Gene_ChrB exactly as written
# "unordered" = Gene pair collapsed regardless of order
pair_mode = "unordered"

# ============================================================
# HELPERS
# ============================================================
def clean_gene(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "" or x.upper() == "NA":
        return None
    return x.upper()

def make_ordered_pair(a, b):
    if a is None or b is None:
        return None
    return f"{a}--{b}"

def make_unordered_pair(a, b):
    if a is None or b is None:
        return None
    return "--".join(sorted([a, b]))

def split_samples(x):
    """
    Split Sample_List into individual sample IDs.
    Handles comma-separated sample lists and removes duplicates.
    """
    if pd.isna(x):
        return []
    seen = set()
    out = []
    for s in str(x).split(","):
        s = s.strip()
        if s and s.upper() != "NA" and s not in seen:
            seen.add(s)
            out.append(s)
    return out

def merge_unique(series):
    seen = set()
    merged = []
    for x in series.dropna():
        x = str(x).strip()
        if x and x not in seen:
            seen.add(x)
            merged.append(x)
    return ",".join(merged)

# ============================================================
# LOAD
# ============================================================
wgs = pd.read_csv(wgs_file, sep="\t")

wgs["Gene_ChrA_clean"] = wgs["Gene_ChrA"].apply(clean_gene)
wgs["Gene_ChrB_clean"] = wgs["Gene_ChrB"].apply(clean_gene)

# keep only rows where both genes are available
wgs_valid = wgs.dropna(subset=["Gene_ChrA_clean", "Gene_ChrB_clean"]).copy()

# ============================================================
# PAIR LABEL
# ============================================================
if pair_mode == "ordered":
    wgs_valid["pair"] = wgs_valid.apply(
        lambda x: make_ordered_pair(x["Gene_ChrA_clean"], x["Gene_ChrB_clean"]), axis=1
    )
elif pair_mode == "unordered":
    wgs_valid["pair"] = wgs_valid.apply(
        lambda x: make_unordered_pair(x["Gene_ChrA_clean"], x["Gene_ChrB_clean"]), axis=1
    )
else:
    raise ValueError("pair_mode must be 'ordered' or 'unordered'")

# ============================================================
# COLLAPSE TO UNIQUE PAIR + SAMPLE
# ============================================================
# Convert Sample_List into individual sample IDs
wgs_valid["sample_id"] = wgs_valid["Sample_List"].apply(split_samples)

# Expand rows if Sample_List contains multiple comma-separated samples
wgs_expanded = wgs_valid.explode("sample_id").copy()

# Remove rows with missing sample_id
wgs_expanded = wgs_expanded.dropna(subset=["sample_id"]).copy()
wgs_expanded["sample_id"] = wgs_expanded["sample_id"].astype(str).str.strip()

# Keep only one record per pair per sample
wgs_pair_sample_unique = wgs_expanded.drop_duplicates(subset=["pair", "sample_id"]).copy()

# ============================================================
# SUMMARY
# ============================================================
raw_total_events = len(wgs_valid)                       # original row count
unique_pair_sample_events = len(wgs_pair_sample_unique)  # deduplicated by pair+sample
unique_pairs = wgs_pair_sample_unique["pair"].nunique()

# pair summary: count = number of UNIQUE samples carrying that pair
pair_counts = (
    wgs_pair_sample_unique
    .groupby("pair", as_index=False)
    .agg(
        count=("sample_id", "size"),
        sample=("sample_id", merge_unique)
    )
)

# optional: keep raw row count for reference
raw_pair_counts = (
    wgs_valid
    .groupby("pair", as_index=False)
    .agg(raw_row_count=("pair", "size"))
)

pair_counts = (
    pair_counts
    .merge(raw_pair_counts, on="pair", how="left")
    .sort_values(["count", "raw_row_count", "pair"], ascending=[False, False, True])
)

dup_pairs = pair_counts[pair_counts["count"] > 1].copy()
n_dup_pair_types = len(dup_pairs)

print("============================================================")
print(f"Pair mode used:                       {pair_mode}")
print(f"Raw valid WGS rows:                   {raw_total_events}")
print(f"Unique pair+sample events:            {unique_pair_sample_events}")
print(f"Unique {pair_mode} pairs:             {unique_pairs}")
print(f"{pair_mode.capitalize()} pairs repeated >1 sample: {n_dup_pair_types}")
print("============================================================")

if dup_pairs.empty:
    print(f"No duplicated {pair_mode} pairs found across different samples.")
else:
    print(f"\nDuplicated {pair_mode} pairs (count = unique samples):")
    print(dup_pairs[["pair", "count", "raw_row_count", "sample"]].to_string(index=False))

# ============================================================
# SAVE ONLY ONE FILE = full rows for duplicated pairs
# ============================================================
if not dup_pairs.empty:
    dup_pair_set = set(dup_pairs["pair"])
    dup_rows = wgs_valid[wgs_valid["pair"].isin(dup_pair_set)].copy()
    dup_rows = dup_rows.merge(dup_pairs, on="pair", how="left")
    dup_rows = dup_rows.sort_values(
        ["count", "raw_row_count", "pair", "ChrA", "PositionA", "ChrB", "PositionB"],
        ascending=[False, False, True, True, True, True, True]
    )

    out_file = os.path.join(outdir, f"WGS_duplicated_{pair_mode}_pairs.tsv")
    dup_rows.to_csv(out_file, sep="\t", index=False)

    print("\nSaved duplicated rows to:")
    print(out_file)

# ============================================================
# OPTIONAL: save full pair summary
# ============================================================
summary_file = os.path.join(outdir, f"WGS_{pair_mode}_pair_summary.tsv")
pair_counts.to_csv(summary_file, sep="\t", index=False)

print("\nSaved pair summary to:")
print(summary_file)

Pair mode used:                       unordered
Raw valid WGS rows:                   95
Unique pair+sample events:            89
Unique unordered pairs:             89
Unordered pairs repeated >1 sample: 0
No duplicated unordered pairs found across different samples.

Saved pair summary to:
/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Delly/Delly_ANNOT_GTF/WGS_pair_summary/WGS_unordered_pair_summary.tsv


In [20]:
# Check the summary of Lumpy_BND
import os
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
wgs_file = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Lumpy/Lumpy_ANNOT_GTF/ALL_SAMPLES.BND.pairs.tsv"
outdir = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Lumpy/Lumpy_ANNOT_GTF/WGS_pair_summary"
os.makedirs(outdir, exist_ok=True)

# Choose one:
# "ordered"   = Gene_ChrA--Gene_ChrB exactly as written
# "unordered" = Gene pair collapsed regardless of order
pair_mode = "unordered"

# ============================================================
# HELPERS
# ============================================================
def clean_gene(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "" or x.upper() == "NA":
        return None
    return x.upper()

def make_ordered_pair(a, b):
    if a is None or b is None:
        return None
    return f"{a}--{b}"

def make_unordered_pair(a, b):
    if a is None or b is None:
        return None
    return "--".join(sorted([a, b]))

def split_samples(x):
    """
    Split Sample_List into individual sample IDs.
    Handles comma-separated sample lists and removes duplicates.
    """
    if pd.isna(x):
        return []
    seen = set()
    out = []
    for s in str(x).split(","):
        s = s.strip()
        if s and s.upper() != "NA" and s not in seen:
            seen.add(s)
            out.append(s)
    return out

def merge_unique(series):
    seen = set()
    merged = []
    for x in series.dropna():
        x = str(x).strip()
        if x and x not in seen:
            seen.add(x)
            merged.append(x)
    return ",".join(merged)

# ============================================================
# LOAD
# ============================================================
wgs = pd.read_csv(wgs_file, sep="\t")

wgs["Gene_ChrA_clean"] = wgs["Gene_ChrA"].apply(clean_gene)
wgs["Gene_ChrB_clean"] = wgs["Gene_ChrB"].apply(clean_gene)

# keep only rows where both genes are available
wgs_valid = wgs.dropna(subset=["Gene_ChrA_clean", "Gene_ChrB_clean"]).copy()

# ============================================================
# PAIR LABEL
# ============================================================
if pair_mode == "ordered":
    wgs_valid["pair"] = wgs_valid.apply(
        lambda x: make_ordered_pair(x["Gene_ChrA_clean"], x["Gene_ChrB_clean"]), axis=1
    )
elif pair_mode == "unordered":
    wgs_valid["pair"] = wgs_valid.apply(
        lambda x: make_unordered_pair(x["Gene_ChrA_clean"], x["Gene_ChrB_clean"]), axis=1
    )
else:
    raise ValueError("pair_mode must be 'ordered' or 'unordered'")

# ============================================================
# COLLAPSE TO UNIQUE PAIR + SAMPLE
# ============================================================
# Convert Sample_List into individual sample IDs
wgs_valid["sample_id"] = wgs_valid["Sample_List"].apply(split_samples)

# Expand rows if Sample_List contains multiple comma-separated samples
wgs_expanded = wgs_valid.explode("sample_id").copy()

# Remove rows with missing sample_id
wgs_expanded = wgs_expanded.dropna(subset=["sample_id"]).copy()
wgs_expanded["sample_id"] = wgs_expanded["sample_id"].astype(str).str.strip()

# Keep only one record per pair per sample
wgs_pair_sample_unique = wgs_expanded.drop_duplicates(subset=["pair", "sample_id"]).copy()

# ============================================================
# SUMMARY
# ============================================================
raw_total_events = len(wgs_valid)
unique_pair_sample_events = len(wgs_pair_sample_unique)
unique_pairs = wgs_pair_sample_unique["pair"].nunique()

# pair summary: count = number of UNIQUE samples carrying that pair
pair_counts = (
    wgs_pair_sample_unique
    .groupby("pair", as_index=False)
    .agg(
        count=("sample_id", "size"),
        sample=("sample_id", merge_unique)
    )
    .sort_values(["count", "pair"], ascending=[False, True])
)

dup_pairs = pair_counts[pair_counts["count"] > 1].copy()
n_dup_pair_types = len(dup_pairs)

print("============================================================")
print(f"Pair mode used:                       {pair_mode}")
print(f"Raw valid WGS rows:                   {raw_total_events}")
print(f"Unique pair+sample events:            {unique_pair_sample_events}")
print(f"Unique {pair_mode} pairs:             {unique_pairs}")
print(f"{pair_mode.capitalize()} pairs repeated >1 sample: {n_dup_pair_types}")
print("============================================================")

if dup_pairs.empty:
    print(f"No duplicated {pair_mode} pairs found across different samples.")
else:
    print(f"\nTop 5 duplicated {pair_mode} pairs (count = unique samples):")
    print(dup_pairs[["pair", "count", "sample"]].head(5).to_string(index=False))

# ============================================================
# SAVE ONLY ONE FILE = full rows for duplicated pairs
# ============================================================
if not dup_pairs.empty:
    dup_pair_set = set(dup_pairs["pair"])
    dup_rows = wgs_valid[wgs_valid["pair"].isin(dup_pair_set)].copy()
    dup_rows = dup_rows.merge(dup_pairs, on="pair", how="left")
    dup_rows = dup_rows.sort_values(
        ["count", "pair", "ChrA", "PositionA", "ChrB", "PositionB"],
        ascending=[False, True, True, True, True, True]
    )

    out_file = os.path.join(outdir, f"WGS_duplicated_{pair_mode}_pairs.tsv")
    dup_rows.to_csv(out_file, sep="\t", index=False)

    print("\nSaved duplicated rows to:")
    print(out_file)

# ============================================================
# save full pair summary
# ============================================================
summary_file = os.path.join(outdir, f"WGS_{pair_mode}_pair_summary.tsv")
pair_counts.to_csv(summary_file, sep="\t", index=False)

print("\nTop 5 rows of pair summary:")
print(pair_counts[["pair", "count", "sample"]].head(5).to_string(index=False))

Pair mode used:                       unordered
Raw valid WGS rows:                   18694
Unique pair+sample events:            8642
Unique unordered pairs:             4287
Unordered pairs repeated >1 sample: 1496

Top 5 duplicated unordered pairs (count = unique samples):
                          pair  count                                                                                                                                                                                                                                                 sample
       ADCY10--ENSG00000300745     10 JC287-TP10-705_TP16-M397,JC287-TP16-M241_TP16-M241,JC287-TP14-385_TP14-385,JC287-TP14-490_TP14-490,JC287-TP08-525_TP08-446,JC287-TP17-M11_TP17-M11,JC287-TP16-M953_TP16-M593,JC287-TP09-71_TP09-11,JC287-TP08-1447_TP08-1335,JC287-TP15-M509_TP15-M509
                  AKAP13--CCT7     10 JC287-TP10-705_TP16-M397,JC287-TP14-385_TP14-385,JC287-TP14-490_TP14-490,JC287-TP15-M509_TP15-M509,JC287-TP17-M11_T

In [8]:
# Whether any RNA fusion pair (fusions_in_all_3_callers) is also seen in WGS (Delly) as a breakpoint between the same 2 genes
import pandas as pd

# ============================================================
# Input files
# ============================================================
rna_file = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/fusions_in_all_3_callers.tsv"
wgs_file = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/SVs/Delly/Delly_ANNOT_GTF/ALL_SAMPLES.BND.pairs.tsv"

outdir = "/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/RNA_WGS_concordance"

import os
os.makedirs(outdir, exist_ok=True)

# ============================================================
# Helper
# ============================================================
def clean_gene(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "" or x.upper() == "NA":
        return None
    return x.upper()

def make_unordered_pair(a, b):
    if a is None or b is None:
        return None
    return "--".join(sorted([a, b]))

def make_ordered_pair(a, b):
    if a is None or b is None:
        return None
    return f"{a}--{b}"

# ============================================================
# Load RNA fusions
# ============================================================
rna = pd.read_csv(rna_file, sep="\t")

# Split Fusion column
rna[["RNA_GeneA", "RNA_GeneB"]] = rna["Fusion"].str.split("--", n=1, expand=True)

rna["RNA_GeneA"] = rna["RNA_GeneA"].apply(clean_gene)
rna["RNA_GeneB"] = rna["RNA_GeneB"].apply(clean_gene)

rna = rna.dropna(subset=["RNA_GeneA", "RNA_GeneB"]).copy()

rna["RNA_pair_ordered"] = rna.apply(lambda x: make_ordered_pair(x["RNA_GeneA"], x["RNA_GeneB"]), axis=1)
rna["RNA_pair_unordered"] = rna.apply(lambda x: make_unordered_pair(x["RNA_GeneA"], x["RNA_GeneB"]), axis=1)

# Keep unique RNA fusions
rna_unique = rna.drop_duplicates(subset=["RNA_pair_ordered"]).copy()

# ============================================================
# Load WGS BND pairs
# ============================================================
wgs = pd.read_csv(wgs_file, sep="\t")

wgs["Gene_ChrA"] = wgs["Gene_ChrA"].apply(clean_gene)
wgs["Gene_ChrB"] = wgs["Gene_ChrB"].apply(clean_gene)

# Remove rows where either gene is missing
wgs = wgs.dropna(subset=["Gene_ChrA", "Gene_ChrB"]).copy()

wgs["WGS_pair_ordered"] = wgs.apply(lambda x: make_ordered_pair(x["Gene_ChrA"], x["Gene_ChrB"]), axis=1)
wgs["WGS_pair_reverse"] = wgs.apply(lambda x: make_ordered_pair(x["Gene_ChrB"], x["Gene_ChrA"]), axis=1)
wgs["WGS_pair_unordered"] = wgs.apply(lambda x: make_unordered_pair(x["Gene_ChrA"], x["Gene_ChrB"]), axis=1)

# ============================================================
# Concordance checks
# ============================================================
wgs_ordered_set = set(wgs["WGS_pair_ordered"].dropna())
wgs_reverse_set = set(wgs["WGS_pair_reverse"].dropna())
wgs_unordered_set = set(wgs["WGS_pair_unordered"].dropna())

# Strict same-order match
rna_unique["match_same_order"] = rna_unique["RNA_pair_ordered"].isin(wgs_ordered_set)

# Reverse-order match
rna_unique["match_reverse_order"] = rna_unique["RNA_pair_ordered"].isin(wgs_reverse_set)

# Unordered match (recommended)
rna_unique["match_unordered"] = rna_unique["RNA_pair_unordered"].isin(wgs_unordered_set)

# ============================================================
# Pull matching WGS rows for unordered concordant events
# ============================================================
concordant_pairs = set(rna_unique.loc[rna_unique["match_unordered"], "RNA_pair_unordered"])

wgs_concordant = wgs[wgs["WGS_pair_unordered"].isin(concordant_pairs)].copy()
rna_concordant = rna_unique[rna_unique["match_unordered"]].copy()

# Merge RNA and WGS details
merged = pd.merge(
    rna_concordant,
    wgs_concordant,
    left_on="RNA_pair_unordered",
    right_on="WGS_pair_unordered",
    how="left"
)

# ============================================================
# Save outputs
# ============================================================
rna_unique.to_csv(os.path.join(outdir, "RNA_fusions_with_WGS_match_flags.tsv"), sep="\t", index=False)
wgs_concordant.to_csv(os.path.join(outdir, "WGS_rows_matching_RNA_fusions.tsv"), sep="\t", index=False)
merged.to_csv(os.path.join(outdir, "RNA_WGS_concordant_pairs_merged.tsv"), sep="\t", index=False)

# ============================================================
# Summary
# ============================================================
n_rna = rna_unique["RNA_pair_ordered"].nunique()
n_wgs = wgs["WGS_pair_unordered"].nunique()
n_conc = rna_unique["match_unordered"].sum()

print("============================================================")
print(f"Unique RNA fusions:              {n_rna}")
print(f"Unique WGS gene pairs:           {n_wgs}")
print(f"RNA-WGS concordant pairs:        {n_conc}")
print("============================================================")

print("\nConcordant RNA fusion pairs:")
for x in sorted(rna_unique.loc[rna_unique["match_unordered"], "RNA_pair_ordered"].tolist()):
    print(x)

Unique RNA fusions:              71
Unique WGS gene pairs:           89
RNA-WGS concordant pairs:        1

Concordant RNA fusion pairs:
LAMA2--STX17
